# Objective

The objective of this notebook is to develop a semantic retrieval-based chatbot system for handling queries related to specific datasets, such as FAQs (Frequently Asked Questions) and airport-related data. The focus is on building an efficient backend that uses vector-based search techniques for semantic search and information retrieval, enabling the chatbot to provide relevant and accurate responses to user queries.

# Import Libraries

Import necessary libraries and modules, such as OpenAI for language models, FAISS for vector storage, and other utilities for data processing and handling warnings.

In [1]:
%pip install openai langchain langchain-community langchain-openai faiss-cpu textblob litellm


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
from openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.vectorstores import FAISS
from langchain_community.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI 
from langchain_openai import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from typing import Dict, Any
import json
import os
from textblob import TextBlob
from litellm import completion
import litellm
from litellm import speech
from litellm import transcription, speech
from pathlib import Path
import os

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configuration

In [30]:
# Configuration
LITELLM_API_KEY = "sk-" 

In [11]:
from litellm import completion
import os

## set ENV variables
os.environ["OPENAI_API_KEY"] = LITELLM_API_KEY

response = completion(
  model="gpt-3.5-turbo",
  messages=[{ "content": "Why botvinnik is said to have revolutioned chess in soviet","role": "user"}]
)

print(response)

ModelResponse(id='chatcmpl-BSf5eVgmcRWAEImcx2wQTfqM1dq49', created=1746170834, model='gpt-3.5-turbo-0125', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="Mikhail Botvinnik is considered to have revolutionized chess in the Soviet Union due to several reasons:\n\n1. Development of the Soviet School of Chess: Botvinnik was a key figure in the creation of the Soviet School of Chess, which emphasized intense training, disciplined preparation, and a strong focus on opening theory. This approach revolutionized the way Soviet players approached the game and led to great success on the international stage.\n\n2. Methodical Approach to Chess: Botvinnik was known for his methodical and scientific approach to chess, emphasizing the importance of analysis, preparation, and strategy. His approach influenced generations of Soviet players and helped to elevate the level of chess in the country.\n\n3. Role as a Mentor and Coac

In [31]:
OPENAI_API_KEY = "sk-"

# Load and Preprocess Data

In [13]:

# Load the JSON data
json_file = "./metadata/airport_data.json"  
with open(json_file, 'r') as f:
    data = json.load(f)

# Combine each object into a chunk
chunks = [
    " ".join([f"{key}: {value}" for key, value in item.items() if value is not None])
    for item in data
]

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OPENAI_API_KEY) 

# Create FAISS index with the chunks
vectorstore = FAISS.from_texts(chunks, embedding=embeddings)

# Save the FAISS index locally for later use
vectorstore.save_local("faiss_airport_data")

# Initialize Components

In [14]:
FAISS_FAQ_PATH = "faiss_airport_data"
litellm.api_key = LITELLM_API_KEY

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", 
    openai_api_key=LITELLM_API_KEY
)

faq_vectorstore = FAISS.load_local(
    FAISS_FAQ_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=LITELLM_API_KEY)

# Update the qa_prompt to remove sentiment
qa_prompt = PromptTemplate(
    template="""
    Answer the following question based on the provided context:

    Context:
    {context}

    Question: {question}

    Instructions:
    - Provide clear and accurate information from the context
    - Format responses neatly and include examples when helpful
    - Be polite and professional
    - Do not mention being an AI

    Answer:""",
    input_variables=["context", "question"]
)

# Sentiment Analysis


This function checks the sentiment of a text and labels it as "positive," "negative," or "neutral." It first looks for specific negative words or phrases to catch rude or frustrated tones. If no negative patterns are found, it uses TextBlob to analyze the sentiment and adjusts the sensitivity for more accurate results.

In [15]:

def detect_sentiment(query: str) -> str:
    """
    Detect the sentiment of the query with improved handling of rude/frustrated tones.
    Returns: "positive", "negative", or "neutral"
    """
    try:
        # First, check for common negative patterns
        negative_patterns = [
            "dont", "don't", "not", "wrong", "bad", "terrible",
            "why dont", "why don't", "why can't", "why do you",
            "stupid", "useless", "waste", "annoying"
        ]
        
        query_lower = query.lower()
        
        # Check for negative patterns first
        for pattern in negative_patterns:
            if pattern in query_lower:
                return "negative"
        
        # If no negative patterns, use TextBlob for sentiment
        analysis = TextBlob(query)
        
        # Adjust thresholds to be more sensitive to negative sentiment
        if analysis.sentiment.polarity > 0.1:  # Increased threshold for positive
            return "positive"
        elif analysis.sentiment.polarity < 0:  # More sensitive to negative
            return "negative"
        else:
            return "neutral"
            
    except Exception as e:
        print(f"Error in sentiment detection: {e}")
        return "neutral"  # Default fallback


# Classification Query

In [16]:

def classify_query(query: str) -> str:
    """Classify query into flight or FAQ category."""
    try:
        response = completion(
            model="gpt-4o",
            messages=[{
                "role": "user",
                "content": f"""
                 Classify the following query into one of the categories:
        1. flight - If the query is related to booking a flight, such as asking about flight availability, departure or arrival timings, or traveling from one place to another by flight.
        2. faq - For all other types of queries, including those related to stores, food, coffee, shopping, or general airport information like facilities, services like lounge, or directions.

        Query: {query}

        Respond only with the category name: flight or faq.
                """
            }],
            temperature=0
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        return "faq"  # Default fallback


In [17]:

def detect_input_language(text: str) -> str:
    """
    Detect the language of input text using LiteLLM.
    Returns the language code (e.g., 'en', 'hi', 'ta', etc.)
    """
    try:
        language_detection_prompt = f"""
        Detect the language of this text: "{text}"
        Response format: Return ONLY the ISO language code (e.g., 'en' for English, 'hi' for Hindi, 'ta' for Tamil, etc.)
        If unsure, return 'en'.
        """
        
        response = completion(
            model="gpt-4o",
            messages=[{"role": "user", "content": language_detection_prompt}],
            temperature=0
        )
        
        detected_lang = response.choices[0].message.content.strip().lower()
        return detected_lang
        
    except Exception as e:
        return "en"  # Default to English on error

def process_text_query(query: str) -> str:
    """Process text queries with language detection and translation."""
    try:
        # Detect input language
        input_language = detect_input_language(query)
        
        # If not English, translate to English first
        if input_language != "en":
            query_in_english = translate_text(query, "English")
        else:
            query_in_english = query
            
        # Get classification and sentiment for the English query
        classification = classify_query(query_in_english)
        sentiment = detect_sentiment(query_in_english)
        

        # Process the query and get response in English
        if classification == "flight":
            response_in_english = handle_flight_query(query_in_english, sentiment)
        else:
            response_in_english = handle_vectorstore_query(query_in_english, faq_vectorstore, sentiment)
            
        # If original query wasn't in English, translate response back
        if input_language != "en":
            final_response = translate_text(response_in_english, input_language)
        else:
            final_response = response_in_english
            
        return final_response
            
    except Exception as e:
        return "I apologize, but I encountered an error. Could you please try asking your question again?"


# Audio Processing

In [18]:

def process_audio(audio_data: bytes) -> Dict[str, Any]:
    """
    Process audio input with full pipeline using LiteLLM:
    1. Transcribe audio
    2. Process query
    3. Translate response
    4. Convert to speech
    """
    temp_path = "temp_audio.wav"
    try:
        # Write audio data to temporary file
        with open(temp_path, "wb") as f:
            f.write(audio_data)

        # Detect language and transcribe using LiteLLM
        with open(temp_path, "rb") as audio_file:
            # Get transcription with language detection
            transcription_response = transcription(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json"
            )
            language = transcription_response.get("language", "en")
            transcription_text = transcription_response.get("text", "")
            
        print(f"Detected Language: {language}")
        
        # Process the transcribed text
        response = process_text_query(transcription_text)
        
        # Translate response if needed
        if language != 'en':
            translated_response = translate_text(response, language)
        else:
            translated_response = response
            
        # Convert to speech using LiteLLM
        output_filename = f"output_{hash(translated_response)}.mp3"
        speech_file_path = Path(output_filename)
        
        speech_response = speech(
            model="openai/tts-1",
            voice="alloy",
            input=translated_response
        )
        speech_response.stream_to_file(speech_file_path)
        
        return {
            "text": transcription_text,
            "language": language,
            "response": response,
            "translated_response": translated_response,
            "audio_response_path": str(speech_file_path)
        }

    except Exception as e:
        print(f"Error processing audio: {e}")
        return {
            "text": "",
            "language": "",
            "response": "I apologize, but I encountered an error processing your audio.",
            "translated_response": "",
            "audio_response_path": ""
        }
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


In [19]:

def detect_language(audio_path: str) -> str:
    """Detect the language of an audio file using LiteLLM."""
    try:
        with open(audio_path, "rb") as audio_file:
            response = transcription(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json"
            )
            return response.get("language", "en")
    except Exception as e:
        return "en"


In [20]:

def translate_text(text: str, target_language: str) -> str:
    """Translate text to target language with Indian context."""
    try:
        translation_response = completion(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": f"""You are a professional Indian translator. 
                    Translate the following text to {target_language} using Indian accent and style.
                    If translating to English, ensure it maintains Indian English characteristics and expressions.
                    For other languages, use vocabulary and expressions commonly used in Indian contexts.
                    Focus on:
                    - Using Indian cultural context and expressions
                    - Maintaining formal yet warm Indian communication style
                    - Using terms and phrases familiar to Indian audience
                    
                    Respond with only the translated text."""
                },
                {"role": "user", "content": text}
            ]
        )
        return translation_response.choices[0].message.content.strip()
    except Exception as e:
        return text


In [21]:

def text_to_speech(text: str, voice: str = "alloy") -> str:
    """Convert text to speech using OpenAI's TTS through LiteLLM."""
    try:
        output_path = f"output_{hash(text)}.mp3"
        
        # Use LiteLLM to route the request
        completion_response = speech(
            model="openai/tts-1",
            messages=[{"role": "user", "content": text}],
            voice=voice,
            api_key=LITELLM_API_KEY
        )
        
        # Save the audio response
        with open(output_path, 'wb') as f:
            f.write(completion_response.audio_content)
            
        return output_path
        
    except Exception as e:
        print(f"Text-to-speech error: {e}")
        return ""


# Handling flight query 

In [23]:

def load_flight_details(file_path='flight_details.json'):
    """
    Load flight details from a JSON file.
    
    Args:
        file_path (str, optional): Path to the JSON file. Defaults to 'flight_details.json'.
    
    Returns:
        list: List of flight details
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
            return data.get('flights', [])
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in {file_path}")
        return []

def extract_flight_information(query: str) -> dict:
    """
    Use advanced LLM-based extraction to precisely identify flight query intent and details.
    
    Args:
        query (str): User's input query
    
    Returns:
        dict: Structured flight information extraction
    """
    try:
        # Comprehensive extraction prompt with intent-focused parsing
        extraction_prompt = f"""
        FLIGHT QUERY ANALYSIS INSTRUCTIONS:

        Analyze the following query with EXTREME PRECISION to understand the user's intent:
        Query: "{query}"

        CRITICAL EXTRACTION OBJECTIVES:
        1. DETERMINE THE EXACT INTENT OF THE QUERY
           - Is the user looking for flights to/from a specific place?
           - Are they seeking flight details, status, or general information?
           - What specific information are they requesting?

        2. EXTRACT PRECISE ROUTE AND LOCATION INFORMATION
           - Identify ALL mentioned cities or locations
           - Determine potential origin and destination
           - Consider context and implied locations

        3. IDENTIFY SPECIFIC QUERY CHARACTERISTICS
           - Flight numbers
           - Specific flight status
           - Dates or time-related information

        RESPONSE REQUIREMENTS:
        Provide a STRICT JSON response with the following structure:
        {{
            "query_type": "route" or "flight_number" or "city" or "status" or "general",
            "origin_city": "EXACT origin city name (lowercase, if found)",
            "destination_city": "EXACT destination city name (lowercase, if found)",
            "flight_number": "Specific flight number (if mentioned)",
            "additional_context": "Any additional relevant context or implied intent"
        }}

        CRITICAL EXTRACTION RULES:
        - ONLY extract information EXPLICITLY or STRONGLY IMPLIED in the query
        - Use lowercase for all city names
        - If NO clear intent is found, use "general" as query_type
        - BE PRECISE and AVOID FABRICATION

        INTENT CLASSIFICATION GUIDELINES:
        - "route": Query clearly seeks flights between two specific locations
        - "city": Query is about flights to/from a single city
        - "flight_number": Specific flight is requested
        - "status": Looking for flight status information
        - "general": Broad or unclear flight-related query

        Provide your most PRECISE and INTELLIGENT interpretation.
        """

        # Use LLM to extract detailed query information
        response = completion(
            model="gpt-4o",
            messages=[{
                "role": "user", 
                "content": extraction_prompt
            }],
            temperature=0.1  # Extremely low for precision
        )
        
        # Parse and validate the JSON response
        try:
            # Clean and parse the response
            response_text = response.choices[0].message.content.strip()
            
            # Remove any markdown code block formatting
            if response_text.startswith('```json'):
                response_text = response_text.strip('```json').strip('```')
            
            # Parse JSON
            info = json.loads(response_text)
            
            # Validate and clean the extracted information
            if not isinstance(info, dict):
                raise ValueError("Invalid JSON structure")
            
            # Ensure query_type is valid
            valid_query_types = ["route", "flight_number", "city", "status", "general"]
            info["query_type"] = info.get("query_type", "general").lower()
            if info["query_type"] not in valid_query_types:
                info["query_type"] = "general"
            
            # Clean and lowercase city names
            for city_key in ["origin_city", "destination_city"]:
                if info.get(city_key):
                    info[city_key] = info[city_key].lower().strip()
                else:
                    info[city_key] = ""
            
            # Ensure consistent structure
            info.setdefault("flight_number", "")
            info.setdefault("additional_context", "")
            
            # Debug print for verification
            print("Extracted Flight Information:", info)
            
            return info
        
        except (json.JSONDecodeError, ValueError) as json_error:
            return {"query_type": "general", "origin_city": "", "destination_city": "", "flight_number": "", "additional_context": ""}
    
    except Exception as e:
        return {"query_type": "general", "origin_city": "", "destination_city": "", "flight_number": "", "additional_context": ""}

def handle_flight_query(query: str, sentiment: str = "neutral") -> str:
    try:
        # Load flight data
        with open('./metadata/flight_details.json', 'r') as file:
            data = json.load(file)
            flights = data.get('flights', [])
        
        # Extract query information
        query_info = extract_flight_information(query)
        
        # Sophisticated matching logic
        matched_flights = []
        
        # Flight Number Exact Match
        if query_info["query_type"] == "flight_number":
            matched_flights = [
                f for f in flights 
                if f["flight_number"].upper() == query_info.get("flight_number", "").upper()
            ]
        
        # Route Matching with Enhanced Flexibility
        elif query_info["query_type"] in ["route", "city"]:
            # Flexible matching that considers partial information
            matched_flights = [
                f for f in flights 
                if ((not query_info.get("origin_city") or 
                     query_info["origin_city"] in f["departure"]["city"].lower()) and
                    (not query_info.get("destination_city") or 
                     query_info["destination_city"] in f["arrival"]["city"].lower()))
            ]
        
        # Status Matching
        elif query_info["query_type"] == "status":
            matched_flights = [
                f for f in flights 
                if f["status"].lower() == query_info.get("status", "").lower()
            ]
        
        # General Query - Return all flights with LLM-generated context
        else:
            matched_flights = flights
        
        # No matches handling with contextual response
        if not matched_flights:
            no_match_response = completion(
                model="gpt-4o",
                messages=[{
                    "role": "user",
                    "content": f"Generate a friendly, helpful response for no flights found matching the query: '{query}'"
                }],
                temperature=0.2
            )
            return no_match_response.choices[0].message.content.strip()
        
        # Format flight details with LLM-enhanced presentation
        flight_details = []
        for flight in matched_flights:
            detail = (
                f"Flight {flight['flight_number']} by {flight['airline']}\n"
                f"From: {flight['departure']['airport']} ({flight['departure']['city']})\n"
                f"To: {flight['arrival']['airport']} ({flight['arrival']['city']})\n"
                f"Departure: {flight['departure_time']}\n"
                f"Arrival: {flight['arrival_time']}\n"
                f"Status: {flight['status']}"
            )
            flight_details.append(detail)
        
        # Generate contextual wrapper
        wrapper_response = completion(
            model="gpt-4o",
            messages=[{
                "role": "user",
                "content": f"Generate a natural, contextual opening for {len(matched_flights)} flight results. Respond with just the opening text."
            }],
            temperature=0.2
        )
        
        # Combine and return
        opening = wrapper_response.choices[0].message.content.strip()
        return f"{opening}\n\n" + "\n\n".join(flight_details)
    
    except Exception as e:
        return "I apologize, but I couldn't process your flight query. Could you please provide more specific details?"


# RAG

In [24]:

def add_empathy(base_response: str, query: str, sentiment: str) -> str:
    """Add appropriate empathy to the response based on sentiment and query content."""

    empathy_prompt = f"""
    Context:
    Original user query: {query}
    Current response: {base_response}
    Sentiment: {sentiment}

    Make this response sound natural and conversational, like helping a friend. Consider:
    
    If they're frustrated:
    - Quick acknowledgment, then straight to helping
    - Keep it simple and direct
    - Focus on solutions

    If they're neutral:
    - Just be friendly and clear
    - Give information straightforwardly
    - Add a light helpful tone

    If they're happy:
    - Keep their upbeat mood going
    - Be enthusiastic but not over-the-top
    - Stay natural and informative

    Important:
    - Keep all facts and details exactly as they are
    - Sound like a real person talking
    - No customer service jargon
    - No forced positivity
    - Keep the same information, just make it more conversational

    New response:

    """
    
    try:
        response = completion(
            model="gpt-4o",
            messages=[{
                "role": "user",
                "content": empathy_prompt
            }],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return base_response

      
def handle_vectorstore_query(query: str, vectorstore: FAISS, sentiment: str) -> str:
    """Handle queries using a vector store with enhanced formatting and cab booking link."""
    try:
        # Get relevant documents and get base response as before
        retrieved_docs = vectorstore.similarity_search(query)
        context = "\n".join([doc.page_content for doc in retrieved_docs])
        
        # Check if query is cab-related
        cab_keywords = [
            "cab", "taxi", "ride", "transport", "transportation", 
            "book cab", "car service", "pickup", "drop", "travel"
        ]
        
        is_cab_related = any(keyword.lower() in query.lower() for keyword in cab_keywords)
        
        # Prepare QA chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=vectorstore.as_retriever(),
            chain_type="stuff",
            chain_type_kwargs={
                "prompt": qa_prompt
            }
        )
        
        # Get base response
        response = qa_chain.invoke({
            "query": query,
            "question": query,
            "context": context
        })
        
        base_response = response['result'] if isinstance(response, dict) and 'result' in response else str(response)
        
        # Add empathy
        empathetic_response = add_empathy(base_response, query, sentiment)
        
        # Add cab booking link if query is cab-related
        if is_cab_related:
            cab_booking_link = "\n\n🚕 *Book a Cab*\nQuick and convenient cab booking: https://www.ola.com/en-in/airport-cab-service/"
            empathetic_response += cab_booking_link
        
        # Format the response
        return empathetic_response
        
    except Exception as e:
        print(f"Error in QA chain: {str(e)}")
        return ("I understand this might be frustrating. I encountered an error processing your request. Could you please rephrase your question?")
   

# Test the Chatbot

In [25]:

query = "I have been hearing about sakshi gupta a lot who is she"
response = process_text_query(query)
print(f"Response: {response}")

Response: Hey! So, you've been hearing a lot about Sakshi Gupta, huh? She's a really interesting artist and sculptor based in Mumbai. Her work is pretty unique because she uses all sorts of materials like concrete, sand, industrial waste, and metal scrap to create her art. She often explores themes like transformation and the passage of time, which is pretty cool.

Sakshi has a Master's degree in Fine Arts (Sculpture) from the College of Art in New Delhi, which she completed back in 2004. She's had her work showcased in solo and group exhibitions all over the world, in places like New York, Vienna, Mumbai, Riga, Beirut, Singapore, and Paris. In 2020, she even did a virtual residency with the British Council and created an installation for their new London headquarters.

She's received quite a few accolades too, like the Visiting Artist Fellowship at Harvard University in 2019, the Illy Sustain Award in 2011, the Civitella Ranieri Fellowship from 2011 to 2012, and the Inlaks Internation

In [26]:
query = """
Somethings never change in India, 080 Lounge at T2
@BLRAirport
- Priority Pass holders aren’t allowed as it does not work
@prioritypasscom
really, an international lounge does not take your cards #scaminthenameofloungeaccess #t2terminalblrairport#blrairport
"""
response = process_text_query(query)
print(f"Response: {response}")

Response: Hey there! I totally get how frustrating it can be when things don't go as expected, especially when you're traveling. So, here's the scoop: The lounge facilities at BLR Airport, both for international and domestic flights, are run by TFS (Travel and Food Services). If you're flying internationally, you'll find the lounge inside the terminal after you pass through security. You can give them a ring at 8657541675 or 080-46577150 to get more details.

For domestic flights, the setup is pretty similar—just head to the lounge after security. If you need to reach out, their contact numbers are 8657541678 or 080-46577100.

Now, about the Priority Pass issue—unfortunately, the info I have doesn't specify why it's not being accepted. It might be best to call the lounge directly using those numbers to see what's up and maybe find another way to get access. Hope this helps, and safe travels!


In [16]:
query = "ನಾನು BLR ವಿಮಾನ ನಿಲ್ದಾಣದಲ್ಲಿ ನನ್ನ ವಸ್ತುಗಳನ್ನು ಕಳೆದುಕೊಂಡೆ. ಅವುಗಳನ್ನು ಮರಳಿ ಪಡೆಯಲು ಮಾರ್ಗವಿದೆಯೇ?"
response = process_text_query(query)
print(f"Response: {response}")

Response: ಹೇನು ಗುರು,

ನೀವು BLR ವಿಮಾನ ನಿಲ್ದಾಣದಲ್ಲಿ ನಿಮ್ಮ ಸಾಮಾನುಗಳನ್ನು ಕಳೆದುಕೊಂಡಿದ್ದಕ್ಕೆ ವಿಷಾದವಾಯಿತು. ಆದರೆ, ಚಿಂತೆ ಬೇಡ, ಅದನ್ನು ಹಿಂತಿರುಗಿಸಿಕೊಳ್ಳುವ ಒಂದು ಮಾರ್ಗವಿದೆ. ಎಲ್ಲಾ ಕಳೆದುಹೋಗಿದ ವಸ್ತುಗಳು "ಲಾಸ್ಟ್ ಅಂಡ್ ಫೌಂಡ್" ವಿಭಾಗದಲ್ಲಿ ತಲುಪುತ್ತವೆ. ನೀವು ಅವರೊಂದಿಗೆ ಸಂಪರ್ಕಿಸಲು 080 22012001 ಗೆ ಕರೆ ಮಾಡಬಹುದು. ಅಥವಾ ನಿಮಗೆ ಇಮೇಲ್ ಕಳುಹಿಸಲು ಇಷ್ಟವಿದ್ದರೆ, lostandfound@bialairport.com ಅಥವಾ t2lostandfound@bialairport.com ಗೆ ಸಂಪರ್ಕಿಸಬಹುದು.

ಜೊತೆಗೆ, "ಲಗ್ಜೇಜ್ ಸ್ಟೋರೇಜ್" ಸೇವೆಯೊಂದಿಗೂ ನೀವೇನು ತಪಾಸಣೆ ಮಾಡಬಹುದು—ಅವರು "ಲಾಸ್ಟ್ ಅಂಡ್ ಫೌಂಡ್" ಜೊತೆಗೆ ಸಂಭಂಧ ಹೊಂದಿದ್ದು, ನಿಮ್ಮ ಸಾಮಾನುಗಳು ಮರಳಿ ದೊರೆತವರೆಗೆ ನಿಮಗೆ ಇನ್ನಷ್ಟು ಸಹಾಯ ಮಾಡಬಹುದು.

ಇದು ನಿಮಗೆ ಸಹಾಯಕವಾಗಲಿದೆ ಎಂದು ನಂಬಿದ್ದೇನೆ, ಹಾಗೂ ನಿಮ್ಮ ವಸ್ತುಗಳು ಬೇಗ ದೊರೆಯಲೆಂದು ಆಶಿಸುತ್ತೇನೆ!


In [27]:
with open("./audio_input/hindi.mp3", "rb") as audio_file:
    audio_data = audio_file.read()

audio_result = process_audio(audio_data)

print("\nAudio Processing Results:")
print(f"Detected Language: {audio_result['language']}")
print(f"Transcription: {audio_result['text']}")
print(f"\nEnglish Response: {audio_result['response']}")
print(f"Translated Response: {audio_result['translated_response']}")
print(f"Audio Response File: {audio_result['audio_response_path']}")
        

Detected Language: hindi

Audio Processing Results:
Detected Language: hindi
Transcription: मैं एक घंटा पहले पहुँच रहा हूँ क्या कोई फिटनेश सेंटर उपलब्ध है?

English Response: नमस्ते! मैंने एयरपोर्ट पर किसी विशेष फिटनेस सेंटर का उल्लेख नहीं पाया, लेकिन here's what I do know: प्रस्थान और आगमन दोनों के लाउंज में शावर सुविधाएं हैं। अगर आप चाहें तो यह आपके लिए ताजा होने का एक अच्छा तरीका हो सकता है। इसके अलावा, अगर आपके पास कुछ और समय है, तो प्रस्थान क्षेत्र में 080 ट्रांजिट होटल द्वारा प्रबंधित एक रूम सुविधा भी है। अगर आपको और जानकारी चाहिए, तो एयरपोर्ट या 080 ट्रांजिट होटल को एक बार कॉल करना फायदेमंद हो सकता है। आशा है यह मददगार होगा!
Translated Response: Hello! I couldn't find any specific fitness center mentioned at the airport, but here's what I do know: Both the departure and arrival lounges have shower facilities. If you wish, this could be a good way to freshen up. Additionally, if you have some more time, there is also a room facility managed by the 080 Transit Hotel in the departu

In [28]:
with open("./audio_input/marathi.mp3", "rb") as audio_file:
    audio_data = audio_file.read()

audio_result = process_audio(audio_data)

print("\nAudio Processing Results:")
print(f"Detected Language: {audio_result['language']}")
print(f"Transcription: {audio_result['text']}")
print(f"\nEnglish Response: {audio_result['response']}")
print(f"Translated Response: {audio_result['translated_response']}")
print(f"Audio Response File: {audio_result['audio_response_path']}")
        

Detected Language: marathi

Audio Processing Results:
Detected Language: marathi
Transcription: मी एक तास अधी पहुँचना रहा है एर्पोर्टला, तर काई-काई फैसिलिटीज अविलिबल आहे।

English Response: नमस्कार!

तर, तुम्ही विमानतळावर एक तास आधी पोहोचत आहात—बघूया, तिथे असताना तुम्ही काय काय करू शकता.

1. **लाऊंज प्रवेश**: सामान्यतः, तुम्ही विमानाच्या उड्डाणाच्या तीन तासांपूर्वी लाऊंजमध्ये आराम करू शकता. तुम्ही फक्त एक तास आधी आल्यामुळे, सरळ आत जाणे शक्य नसेल. पण जर तुम्हाला तिथे अधिक वेळ थांबायचे असेल, तर रिसेप्शनवरील लोकांशी बोला—ते तुम्हाला मदत करू शकतील.

2. **शारिरीक अपंग प्रवाशांसाठी सुविधा**:
   - वाहनांना सोडण्याची किंवा उचलण्याची सोयीची सुविधा उपलब्ध आहे.
   - विमानतळावर तुम्हाला बग्गी सेवा किंवा व्हीलचेअर मिळू शकते, जर गरज असेल तर.
   - विशेष आसनक्षेत्रे आणि चेक-इन पासून बोर्डिंगपर्यंत प्राधान्य प्रवेश आहे.
   - दृष्टीबाधितांसाठी टॅक्टाइल फरशीची व्यवस्था आहे.

3. **विशेष सेवाएं अपंग प्रवाशांसाठी**:
   - विशेष ड्रॉप-ऑफ आणि पिक-अप क्षेत्रे आहेत.
   - ड्रॉप-ऑफ पॉइंटवर कॉलिंग सुविधा उपलब्ध आहे

In [29]:
with open("./audio_input/marathi2.mp3", "rb") as audio_file:
    audio_data = audio_file.read()

audio_result = process_audio(audio_data)

print("\nAudio Processing Results:")
print(f"Detected Language: {audio_result['language']}")
print(f"Transcription: {audio_result['text']}")
print(f"\nEnglish Response: {audio_result['response']}")
print(f"Translated Response: {audio_result['translated_response']}")
print(f"Audio Response File: {audio_result['audio_response_path']}")
        

Detected Language: marathi

Audio Processing Results:
Detected Language: marathi
Transcription: माला तीन प्रश्ना आहेथ. प्रश्ना एक, माझी बैक साइज लिमिट काई आहे फ्लाइट वर कैरी करनेची? धूस्रा प्रश्ना, जर माझे बिलॉंगिंग्स एर्पोर्ट वर हरवले, तर ते कशे पुनह मिलतील? धूस्रा प्रश्ना, दिल्ली तो बैंगलोर दैरेक फ्लाइट आहे का इवलिबल?

English Response: नमस्कार! तुमच्या प्रश्नांमध्ये मी तुम्हाला मदत करू शकतो:

1. **उड्डाणाच्या दरम्यान सामानाच्या आकाराची मर्यादा:**
   - सामानाच्या आकाराच्या मर्यादेची नेमकी माहिती देणे मला शक्य नाही कारण ती विमानकंपनीनुसार बदलू शकते. सर्वात योग्य माहिती मिळविण्यासाठी तुम्ही ज्या विमानकंपनीसोबत उड्डाण करत आहात त्या कंपनीशी थेट संपर्क करणे उत्तम राहील.

2. **विमानतळावर हरवलेल्या वस्तू कशा शोधाव्यात:**
   - जर तुम्ही विमानतळावर काही हरवले असेल तर काळजी करू नका! विमानतळाच्या हरवलेले आणि सापडलेले विभागाशी किंवा विमानतळ अधिकाऱ्यांशी संपर्क साधा. ते तुमच्या वस्तू शोधण्यात मदत करू शकतील.

3. **दिल्ली ते बंगलोर थेट उड्डाण:**
   - मला ताज्या उड्डाण वेळापत्रकाची माहिती नाही, पण द

# Accuracy

In [19]:
# Ground truth JSON data
ground_truth = [
    {
        "query": "Can I park my Two wheelers at BLR Airport? If yes, what are the charges?",
        "sentiment": "Neutral",
        "response": "Yes, we have an overnight parking facility; for Two-wheelers, it will be INR250 for the first day(24hrs) and every additional day INR 150(24hrs)."
    },
    {
        "query": "Why don't you tell me how to use the lounge at BLR Airport for domestic departures? Or is it just another useless facility?",
        "sentiment": "Negative",
        "response": "Hello, the domestic BLR lounge is located inside the terminal building after the security check, managed by TFS(Travel and Food Services). You may reach out to them on - 8657541678/ 080-46577100"
    },
    {
        "query": "I am very excited and I want to know if Digi yatra is available at T2",
        "sentiment": "Positive",
        "response": "Yes, Digi Yatra is available for passengers travelling domestic sectors."
    },
    {
        "query": "I am fed up of this app and i have a headache",
        "sentiment": "Negative",
        "response": None
    },
    {
        "query": "I am happy to travel from bangalore to chennai and how to book a flight",
        "sentiment": "Positive",
        "response": "Flight SG-267 by SpiceJet From: Kempegowda International Airport (BLR) (Bangalore) To: Chennai International Airport (MAA) (Chennai) Departure: 11:45 AM Arrival: 01:00 PM Status: On Time"
    },
    {
        "query": "WHY DONT YOU RESPOND PROPERLY I NEED A COFFE",
        "sentiment": "Negative",
        "response": None
    },
    {
        "query": "I have been hearing about sakshi gupta a lot may i know who is she",
        "sentiment": "Neutral",
        "response": "The JSON data contains information about Sakshi Gupta, a Mumbai-based contemporary sculptor and artist. Some key details include:\n\n1. Sakshi Gupta ID: 328.\n2. She is a mixed media artist and often works with materials such as concrete, sand, industrial waste, and metal scrap.\n3. Themes in her work often include transformation and processes of time.\n4. She has a Master's in Fine Arts (Sculpture) from College of Art, New Delhi (2004).\n5. She has held various solo and group exhibitions in locations like New York, Vienna, Mumbai, Riga, Beirut, Singapore, and Paris.\n6. She was awarded a virtual residency with the British Council in 2020 where she made an installation for their new London headquarters.\n7. She has received various awards including the Visiting Artist Fellowship at Harvard University (2019), the Illy Sustain Award (2011), the Civitella Ranieri Fellowship (2011-2012), and the Inlaks International Scholarship (2007).\n\nIn addition, there are two images associated with Sakshi Gupta available in PNG format on the 'strapi-provider-upload-aws-s3-advanced' server. They are stored at the specified URLs in either a thumbnail or original version. \n\nThe item was first created at 2023-11-10T05:43:39.152Z and last updated at 2023-11-10T07:21:07.708Z. However, no information on when it was published is provided."
    }
]

# Initialize counters
correct_sentiment_predictions = 0
correct_response_predictions = 0

for data in ground_truth:
    query = data["query"]
    expected_sentiment = data["sentiment"].lower()
    expected_response = data["response"]

    # Run the query through the script
    detected_sentiment = detect_sentiment(query)
    script_response = process_text_query(query)

    # Compare sentiment
    if detected_sentiment == expected_sentiment:
        correct_sentiment_predictions += 1

    # Compare response
    if script_response.strip() == (expected_response.strip() if expected_response else None):
        correct_response_predictions += 1

# Total queries
total_queries = len(ground_truth)

# Calculate accuracies
sentiment_accuracy = (correct_sentiment_predictions / total_queries) * 100

# Display results
print(f"Sentiment Accuracy: {sentiment_accuracy:.2f}%")


Extracted Flight Information: {'query_type': 'route', 'origin_city': 'bangalore', 'destination_city': 'chennai', 'flight_number': '', 'additional_context': 'User is interested in booking a flight from Bangalore to Chennai.'}
Sentiment Accuracy: 85.71%
